In [0]:

---- EXEMPLOS DE UERIES DE PERMISSÃO ----
GRANT USE CATALOG
ON CATALOG projeto2
TO `analista_dados`;

GRANT USE SCHEMA
ON SCHEMA projeto2.gold
TO `analista_dados`;

GRANT SELECT
ON SCHEMA projeto2.gold
TO `analista_dados`;

GRANT SELECT
ON TABLE projeto2.gold.fato_matriculas
TO `analista_dados`;

GRANT SELECT
ON TABLE projeto2.gold.dim_aluno
TO `analista_dados`;

GRANT SELECT
ON TABLE projeto2.gold.dim_curso
TO `analista_dados`;

GRANT SELECT
ON TABLE projeto2.gold.dim_localidade
TO `analista_dados`;

---- GOVERNANÇA ----
CREATE GOVERNED TAG classification
  DESCRIPTION 'Classificação de sensibilidade dos dados'
  VALUES ('green', 'yellow', 'red');

ALTER TABLE projeto2.gold.dim_aluno
ALTER COLUMN cpf
SET TAGS ('classification' = 'red');

ALTER TABLE projeto2.gold.dim_aluno
ALTER COLUMN email
SET TAGS ('classification' = 'yellow');

ALTER TABLE projeto2.gold.dim_aluno
ALTER COLUMN nome
SET TAGS ('classification' = 'yellow');

ALTER TABLE projeto2.gold.dim_aluno
ALTER COLUMN matricula
SET TAGS ('classification' = 'green');

ALTER TABLE projeto2.gold.dim_aluno
ALTER COLUMN matricula
SET TAGS ('classification' = 'green');

CREATE OR REPLACE FUNCTION projeto2.gold.mask_red(cpf STRING)
RETURNS STRING
RETURN '***REDACTED***';

CREATE OR REPLACE FUNCTION projeto2.gold.mask_red(email STRING)
RETURNS STRING
RETURN '***REDACTED***';

CREATE POLICY mask_red_columns
ON SCHEMA projeto2.gold
COLUMN MASK projeto2.gold.mask_red
TO `analista_dados`
FOR TABLES
MATCH COLUMNS has_tag_value('classification', 'red') AS m
ON COLUMN m;

---- MONITORAMENTO DOS JOBS ----

SELECT
    result_state,
    COUNT(*) AS quantidade
FROM system.lakeflow.job_run_timeline
WHERE period_start_time >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY result_state
ORDER BY quantidade DESC;

---- QUERY NA GOLD ----

-- Matrículas ativas por data, UF e curso:
SELECT
    f.data_matricula,
    f.uf,
    f.curso,
    SUM(f.quantidade_matriculas) AS quantidade_matriculas,
    SUM(f.quantidade_alunos) AS quantidade_alunos,
    ROUND(
        SUM(f.media_nota_final * f.quantidade_matriculas)
        / SUM(f.quantidade_matriculas), 2
    ) AS media_nota_final,
    ROUND(
        SUM(f.media_frequencia * f.quantidade_matriculas)
        / SUM(f.quantidade_matriculas), 2
    ) AS media_frequencia
FROM projeto2.gold.fato_matriculas AS f
WHERE f.status = 'ativo'
GROUP BY
    f.data_matricula,
    f.uf,
    f.curso
ORDER BY
    f.data_matricula DESC,
    quantidade_matriculas DESC;
